# CUDA Python: Tasks 1–20
**CSS 314 Parallel Computing · 15 October 2026 · Student ID 230103188**

Select **Runtime → Change runtime type → T4 GPU**, then **Run all**.
This notebook contains the implementations and tests, so it can run without cloning GitHub.
The self-check uses a real CUDA GPU and compares results with NumPy/SciPy references.


In [ ]:
# Environment bootstrap (CUDA 12 libraries work with the T4's driver).
import os, sys, subprocess
from pathlib import Path
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numba-cuda[cu12]', 'numpy', 'scipy', 'pytest'], check=True)
WORK = Path('/content/cuda_tasks_01_20')
WORK.mkdir(exist_ok=True)
os.chdir(WORK)
(WORK / 'lab_kernels').mkdir(exist_ok=True)
(WORK / 'tests').mkdir(exist_ok=True)
print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,driver_version,memory.total', '--format=csv'], text=True))
from numba import cuda
assert os.environ.get('NUMBA_ENABLE_CUDASIM') != '1', 'A real GPU is required.'
assert cuda.is_available(), 'Choose a T4 GPU runtime.'
print('Device:', cuda.get_current_device().name)
print('Compute capability:', cuda.get_current_device().compute_capability)
print('Python:', sys.version)


name, driver_version, memory.total [MiB]
Tesla T4, 580.82.07, 15360 MiB

Device: Tesla T4
Compute capability: ComputeCapability(major=7, minor=5)
Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]


## Implementations
Each module contains named kernels matching the tasksheet and documents its launch geometry.

- **1–8:** vector addition, grid-stride scaling, matrix addition/transpose, volume initialization, grayscale, LERP, threshold.
- **9–17:** memory-access timing, shared stencil/convolution/matmul, padded transpose, sum/argmax reductions, constant FIR, pinned transfers.
- **18–20:** warp shuffle reduction, vote analysis, inclusive Hillis–Steele scan.


In [ ]:
%%writefile lab_kernels/__init__.py
"""CUDA Python lab tasks 1-20. See modules for launch configurations."""


Writing lab_kernels/__init__.py


In [ ]:
%%writefile lab_kernels/module1_execution_model.py
"""Tasks 1–8: Numba CUDA kernels for one, two and three dimensional grids.

Launch the vector kernels with 256 threads, matrices/images with (16, 16),
and ``volume_init_3d`` with (8, 8, 8). Matrix coordinates use CUDA x for
columns and y for rows. Every kernel guards partial final blocks. Inputs and
outputs are device arrays; no host-side numerical fallback is provided.
"""

from numba import cuda, float32


@cuda.jit
def vector_add_1d(a, b, out):
    """Task 01: ``out = a + b`` for equally sized one-dimensional arrays."""
    i = cuda.grid(1)
    if i < out.size:
        out[i] = a[i] + b[i]


@cuda.jit
def vector_scale_grid_stride(x, alpha, out):
    """Task 02: scale an arbitrary length using a grid of any positive size."""
    for i in range(cuda.grid(1), out.size, cuda.gridsize(1)):
        out[i] = x[i] * alpha


@cuda.jit
def matrix_add_2d(A, B, C):
    """Task 03: add matrices, including non-square, non-tile-aligned shapes."""
    col, row = cuda.grid(2)
    if row < C.shape[0] and col < C.shape[1]:
        C[row, col] = A[row, col] + B[row, col]


@cuda.jit
def matrix_transpose_naive(A, B):
    """Task 04: transpose ``A[M,N]`` into ``B[N,M]`` through global memory."""
    col, row = cuda.grid(2)
    if row < A.shape[0] and col < A.shape[1]:
        B[col, row] = A[row, col]


@cuda.jit
def volume_init_3d(vol, D, H, W):
    """Task 05: initialize ``vol[z,y,x] = 10000*z + 100*y + x``.

    With C-order storage its flattened address is ``z*H*W + y*W + x``.
    D, H and W must match the three dimensions of ``vol``.
    """
    x, y, z = cuda.grid(3)
    if z < D and y < H and x < W:
        vol[z, y, x] = z * 10000 + y * 100 + x


@cuda.jit
def rgb_to_grayscale(img_rgb, img_gray):
    """Task 06: BT.601 luminance from RGB uint8 or float32 to float32."""
    x, y = cuda.grid(2)
    if y < img_gray.shape[0] and x < img_gray.shape[1]:
        img_gray[y, x] = (
            float32(0.299) * img_rgb[y, x, 0]
            + float32(0.587) * img_rgb[y, x, 1]
            + float32(0.114) * img_rgb[y, x, 2]
        )


@cuda.jit
def vector_lerp(a, b, t, out):
    """Task 07: elementwise interpolation, using a fused multiply-add."""
    i = cuda.grid(1)
    if i < out.size:
        # fuse a*(1-t) + b*t; this also handles t outside [0, 1].
        out[i] = cuda.fma(float32(1.0) - t[i], a[i], t[i] * b[i])


@cuda.jit
def vector_threshold(x, thresh, mask):
    """Task 08: inclusive threshold into a uint8 or boolean output mask."""
    for i in range(cuda.grid(1), x.size, cuda.gridsize(1)):
        mask[i] = x[i] >= thresh


Writing lab_kernels/module1_execution_model.py


In [ ]:
%%writefile lab_kernels/module2_memory_hierarchy.py
"""Tasks 9–17: shared/constant memory kernels and measured transfer utilities.

The stencil and convolution use 256-thread blocks and fixed shared buffers:
stencil radii 0..32 and convolution filters of 1..65 taps (signal >= filter).
Matmul uses (16,16) blocks; the padded transpose uses (32,8) blocks over
32x32 logical tiles. Reductions support power-of-two blocks <=1024 threads.
All filter operations use float32 and explicit boundary conventions.

Benchmark results are actual CUDA event measurements. Ratios are observations,
not universal guarantees: GPU/cache size, PCIe, payload size and host memory
allocation can change them. No result is fabricated to meet a speedup target.
"""

import numpy as np
from numba import cuda, float32, int64

BLOCK_SIZE = 256
MAX_RADIUS = 32
MAX_FILTER_TAPS = 65
TILE_DIM = 16
TRANSPOSE_TILE = 32
# Fixed, non-symmetric 64-tap causal FIR coefficients, normalized to unit sum.
# Even-sized `same` convolution uses NumPy's left-of-center alignment.
FIR_WEIGHTS = np.exp(-np.arange(64, dtype=np.float32) / np.float32(12.0))
FIR_WEIGHTS = np.ascontiguousarray(FIR_WEIGHTS / FIR_WEIGHTS.sum(), dtype=np.float32)
FIR_WEIGHTS.setflags(write=False)


@cuda.jit
def coalesced_access(in_arr, out_arr):
    """Task 09 baseline: adjacent lanes copy adjacent float32 elements."""
    for i in range(cuda.grid(1), out_arr.size, cuda.gridsize(1)):
        if i < in_arr.size:
            out_arr[i] = in_arr[i]


@cuda.jit
def benchmark_strided_access(in_arr, out_arr, stride):
    """Task 09: ``out[i] = in[i*stride]``; input must cover all accesses.

    A fixed output length keeps the number of useful bytes comparable between
    strides, while the larger input exposes extra memory transactions.
    """
    for i in range(cuda.grid(1), out_arr.size, cuda.gridsize(1)):
        source = i * stride
        if source < in_arr.size:
            out_arr[i] = in_arr[source]


def _event_milliseconds(operation, repeats):
    """Time queued device operations after a separate warm-up launch."""
    operation()
    cuda.synchronize()
    start = cuda.event(timing=True)
    end = cuda.event(timing=True)
    start.record()
    for _ in range(repeats):
        operation()
    end.record()
    end.synchronize()
    return float(cuda.event_elapsed_time(start, end)) / repeats


def profile_strided_access(num_elements=1 << 20, strides=(1, 2, 4, 8, 16, 32), repeats=20):
    """Task 09 host profiler: return useful bandwidth, timings and integrity.

    Effective GB/s counts one 4-byte load and one 4-byte store per output;
    it does not claim to measure physical DRAM transactions or cache misses.
    """
    if num_elements <= 0 or repeats <= 0 or any(s <= 0 for s in strides):
        raise ValueError("positive element count, strides and repeats required")
    results = {}
    blocks = (num_elements + BLOCK_SIZE - 1) // BLOCK_SIZE
    for stride in strides:
        host = np.arange(num_elements * stride, dtype=np.float32)
        device_input = cuda.to_device(host)
        device_output = cuda.device_array(num_elements, dtype=np.float32)
        if stride == 1:
            launch = lambda: coalesced_access[blocks, BLOCK_SIZE](device_input, device_output)
        else:
            launch = lambda: benchmark_strided_access[blocks, BLOCK_SIZE](device_input, device_output, stride)
        ms = _event_milliseconds(launch, repeats)
        results[int(stride)] = {
            "elapsed_ms": ms,
            "effective_gb_s": num_elements * 8 / (ms * 1e6) if ms > 0 else 0.0,
            "integrity": bool(np.array_equal(device_output.copy_to_host(), host[::stride])),
        }
    return results


@cuda.jit
def stencil_1d_shared(in_arr, out_arr, radius):
    """Task 10: moving average with nearest-value edge clamping; radius <=32."""
    # Shared extents must be integer literals for the CUDA compiler.
    tile = cuda.shared.array(320, dtype=float32)
    tid = cuda.threadIdx.x
    width = cuda.blockDim.x
    start = cuda.blockIdx.x * width
    n = in_arr.size
    if n == 0 or radius < 0 or radius > MAX_RADIUS or width > BLOCK_SIZE:
        return
    for offset in range(tid, width + 2 * radius, width):
        source = min(max(start + offset - radius, 0), n - 1)
        tile[offset] = in_arr[source]
    cuda.syncthreads()
    i = start + tid
    if i < n:
        total = float32(0.0)
        for j in range(2 * radius + 1):
            total += tile[tid + j]
        out_arr[i] = total / (2 * radius + 1)


@cuda.jit
def conv_1d_shared_tiled(x, kernel, y):
    """Task 11: zero-padded ``np.convolve(x, kernel, mode='same')``.

    Use <=256 threads and filters of 1..65 elements. Both odd and even tap
    counts are supported, provided ``x.size >= kernel.size``.
    """
    # Shared extents must be integer literals for the CUDA compiler.
    tile = cuda.shared.array(320, dtype=float32)
    weights = cuda.shared.array(MAX_FILTER_TAPS, dtype=float32)
    tid = cuda.threadIdx.x
    width = cuda.blockDim.x
    taps = kernel.size
    start = cuda.blockIdx.x * width
    if taps < 1 or taps > MAX_FILTER_TAPS or width > BLOCK_SIZE:
        return
    for j in range(tid, taps, width):
        weights[j] = kernel[j]
    for offset in range(tid, width + taps - 1, width):
        source = start + offset - taps // 2
        tile[offset] = x[source] if 0 <= source < x.size else float32(0.0)
    cuda.syncthreads()
    i = start + tid
    if i < x.size:
        total = float32(0.0)
        for j in range(taps):
            total += tile[tid + taps - 1 - j] * weights[j]
        y[i] = total


@cuda.jit
def matmul_tiled_shared(A, B, C):
    """Task 12: rectangular ``A[M,K] @ B[K,N]``; launch (16,16) blocks."""
    a_tile = cuda.shared.array((TILE_DIM, TILE_DIM), dtype=float32)
    b_tile = cuda.shared.array((TILE_DIM, TILE_DIM), dtype=float32)
    tx, ty = cuda.threadIdx.x, cuda.threadIdx.y
    col = cuda.blockIdx.x * TILE_DIM + tx
    row = cuda.blockIdx.y * TILE_DIM + ty
    total = float32(0.0)
    for phase in range((A.shape[1] + TILE_DIM - 1) // TILE_DIM):
        ak = phase * TILE_DIM + tx
        bk = phase * TILE_DIM + ty
        a_tile[ty, tx] = A[row, ak] if row < A.shape[0] and ak < A.shape[1] else float32(0.0)
        b_tile[ty, tx] = B[bk, col] if bk < B.shape[0] and col < B.shape[1] else float32(0.0)
        cuda.syncthreads()
        for k in range(TILE_DIM):
            total += a_tile[ty, k] * b_tile[k, tx]
        cuda.syncthreads()
    if row < C.shape[0] and col < C.shape[1]:
        C[row, col] = total


@cuda.jit
def transpose_bank_conflict_free(A, B):
    """Task 13: coalesced transpose with a padded (32,33) tile.

    Launch blocks=(ceil(N/32),ceil(M/32)), threads=(32,8).
    """
    tile = cuda.shared.array((32, 33), dtype=float32)
    tx, ty = cuda.threadIdx.x, cuda.threadIdx.y
    source_col = cuda.blockIdx.x * TRANSPOSE_TILE + tx
    source_row = cuda.blockIdx.y * TRANSPOSE_TILE + ty
    for j in range(0, TRANSPOSE_TILE, 8):
        if source_row + j < A.shape[0] and source_col < A.shape[1]:
            tile[ty + j, tx] = A[source_row + j, source_col]
    cuda.syncthreads()
    output_col = cuda.blockIdx.y * TRANSPOSE_TILE + tx
    output_row = cuda.blockIdx.x * TRANSPOSE_TILE + ty
    for j in range(0, TRANSPOSE_TILE, 8):
        if output_row + j < B.shape[0] and output_col < B.shape[1]:
            B[output_row + j, output_col] = tile[tx, ty + j]


@cuda.jit
def reduction_sum_shared(in_arr, block_sums):
    """Task 14: one sum per power-of-two block of <=1024 threads."""
    scratch = cuda.shared.array(1024, dtype=float32)
    tid = cuda.threadIdx.x
    i = cuda.grid(1)
    scratch[tid] = in_arr[i] if i < in_arr.size else float32(0.0)
    cuda.syncthreads()
    stride = cuda.blockDim.x // 2
    while stride > 0:
        if tid < stride:
            scratch[tid] += scratch[tid + stride]
        cuda.syncthreads()
        stride //= 2
    if tid == 0:
        block_sums[cuda.blockIdx.x] = scratch[0]


@cuda.jit
def reduction_argmax_shared(in_arr, block_max_vals, block_max_indices):
    """Task 15: per-block maximum and earliest int64 global index on ties.

    Launch power-of-two blocks <=1024. NaN ordering is intentionally excluded;
    finite values and infinities are supported, including all -inf input.
    """
    values = cuda.shared.array(1024, dtype=float32)
    indices = cuda.shared.array(1024, dtype=int64)
    tid = cuda.threadIdx.x
    i = cuda.grid(1)
    values[tid] = in_arr[i] if i < in_arr.size else -float('inf')
    indices[tid] = i if i < in_arr.size else 9223372036854775807
    cuda.syncthreads()
    stride = cuda.blockDim.x // 2
    while stride > 0:
        if tid < stride:
            other = tid + stride
            if values[other] > values[tid] or (values[other] == values[tid] and indices[other] < indices[tid]):
                values[tid] = values[other]
                indices[tid] = indices[other]
        cuda.syncthreads()
        stride //= 2
    if tid == 0:
        block_max_vals[cuda.blockIdx.x] = values[0]
        block_max_indices[cuda.blockIdx.x] = indices[0]


@cuda.jit
def constant_memory_filter(signal, out):
    """Task 16: 64-tap fixed FIR in constant memory, NumPy same alignment."""
    weights = cuda.const.array_like(FIR_WEIGHTS)
    for i in range(cuda.grid(1), signal.size, cuda.gridsize(1)):
        total = float32(0.0)
        for j in range(64):
            source = i + 31 - j
            if 0 <= source < signal.size:
                total += signal[source] * weights[j]
        out[i] = total


def benchmark_pinned_transfer(size_bytes, repeats=7):
    """Task 17: measured H2D/D2H transfers with pageable and pinned buffers.

    ``size_bytes`` must be a positive multiple of four. CUDA events time only
    copies using preallocated GPU buffers; allocation/compilation is excluded.
    Results include directional GB/s, timings, copy integrity and observed
    pinned/pageable ratios. No minimum hardware-dependent ratio is asserted.
    """
    if size_bytes <= 0 or size_bytes % 4 or repeats <= 0:
        raise ValueError("size_bytes must be a positive multiple of 4; repeats > 0")
    n = size_bytes // 4
    pageable = np.arange(n, dtype=np.float32)
    pageable_out = np.empty_like(pageable)
    pinned = cuda.pinned_array(n, dtype=np.float32)
    pinned_out = cuda.pinned_array(n, dtype=np.float32)
    pinned[:] = pageable
    device = cuda.device_array(n, dtype=np.float32)
    result = {"size_bytes": int(size_bytes), "repeats": int(repeats)}
    # Pageable transfers may be synchronous in the driver. Events still
    # capture stream work; pinned transfers use an explicit asynchronous stream.
    for label, source, destination in (("pageable", pageable, pageable_out), ("pinned", pinned, pinned_out)):
        stream = cuda.stream()
        device.copy_to_device(source, stream=stream)
        stream.synchronize()
        directions = {}
        for direction in ("h2d", "d2h"):
            samples = []
            for _ in range(repeats):
                start, end = cuda.event(timing=True), cuda.event(timing=True)
                start.record(stream)
                if direction == "h2d":
                    device.copy_to_device(source, stream=stream)
                else:
                    device.copy_to_host(destination, stream=stream)
                end.record(stream)
                end.synchronize()
                samples.append(float(cuda.event_elapsed_time(start, end)))
            ms = float(np.median(samples))
            directions[direction] = {"elapsed_ms": ms, "gb_s": size_bytes / (ms * 1e6) if ms > 0 else 0.0}
        directions["integrity"] = bool(np.array_equal(destination, source))
        result[label] = directions
    result["speedup"] = {
        direction: result["pinned"][direction]["gb_s"] / result["pageable"][direction]["gb_s"]
        if result["pageable"][direction]["gb_s"] > 0 else 0.0
        for direction in ("h2d", "d2h")
    }
    return result


Writing lab_kernels/module2_memory_hierarchy.py


In [ ]:
%%writefile lab_kernels/module3_warp_atomics.py
"""Tasks 18–20: warp intrinsics and a shared-memory inclusive scan.

Launch tasks 18 and 19 with a 1D block size divisible by 32, at most 1024.
Task 18 writes one sum per block. Its warp reductions use only shuffles;
combining the warps uses 32 shared scalars and one block barrier.

Task 19 accepts either a uint32 vector containing one ballot per warp, or
a uint32 matrix with three columns: ballot, all-positive, any-positive.
Padding lanes do not participate in the logical all/any result.

Task 20 is a single-block scan: launch one block of at least len(x) threads,
with at most 1024 threads. The input length need not be a power of two.
"""

from numba import cuda, float64


@cuda.jit
def warp_reduce_sum_shfl(in_arr, out_arr):
    """Task 18: shuffle each warp, then reduce warp sums in the first warp."""
    warp_sums = cuda.shared.array(32, dtype=float64)
    tid = cuda.threadIdx.x
    lane = tid % 32
    warp = tid // 32
    index = cuda.grid(1)
    value = float64(0.0)
    if index < in_arr.size:
        value = float64(in_arr[index])
    for delta in (16, 8, 4, 2, 1):
        value += cuda.shfl_down_sync(0xFFFFFFFF, value, delta)
    if lane == 0:
        warp_sums[warp] = value
    cuda.syncthreads()

    if warp == 0:
        value = float64(0.0)
        if lane < cuda.blockDim.x // 32:
            value = warp_sums[lane]
        for delta in (16, 8, 4, 2, 1):
            value += cuda.shfl_down_sync(0xFFFFFFFF, value, delta)
        if lane == 0:
            out_arr[cuda.blockIdx.x] = value


@cuda.jit
def warp_vote_analyzer(data, mask_out):
    """Task 19: analyze the positive predicate, including a partial final warp."""
    index = cuda.grid(1)
    valid = index < data.size
    positive = False
    if valid:
        positive = data[index] > 0
    ballot = cuda.ballot_sync(0xFFFFFFFF, positive)
    all_positive = cuda.all_sync(0xFFFFFFFF, positive or not valid)
    any_positive = cuda.any_sync(0xFFFFFFFF, positive)
    warp_index = index // 32
    if cuda.threadIdx.x % 32 == 0 and warp_index < (data.size + 31) // 32:
        if mask_out.ndim == 1:
            mask_out[warp_index] = ballot
        else:
            mask_out[warp_index, 0] = ballot
            mask_out[warp_index, 1] = all_positive
            mask_out[warp_index, 2] = any_positive


@cuda.jit
def scan_hillis_steele(x, out):
    """Task 20: inclusive Hillis–Steele scan with shared ping-pong buffers."""
    shared = cuda.shared.array((2, 1024), dtype=float64)
    tid = cuda.threadIdx.x
    value = float64(0.0)
    if tid < x.size:
        value = float64(x[tid])
    shared[0, tid] = value
    cuda.syncthreads()
    source = 0
    stride = 1
    while stride < cuda.blockDim.x:
        target = 1 - source
        value = shared[source, tid]
        if tid >= stride:
            value += shared[source, tid - stride]
        shared[target, tid] = value
        cuda.syncthreads()
        source = target
        stride *= 2
    if tid < x.size:
        out[tid] = shared[source, tid]


Writing lab_kernels/module3_warp_atomics.py


## Automated reference checks
These tests include non-divisible sizes, rectangular matrices, filter boundaries, tie indices, partial warps, and scans up to 1024 elements.
The benchmark tasks measure actual device-event times and verify transferred values. Speedups depend on hardware and are reported without assuming a fixed ratio.


In [ ]:
%%writefile requirements.txt
numba>=0.60
numba-cuda[cu12]>=0.20
numpy>=1.26
scipy>=1.11
pytest>=8.0


Writing requirements.txt


In [ ]:
%%writefile pytest.ini
[pytest]
testpaths = tests
addopts = -ra
filterwarnings =
    ignore:.*Grid size.*:numba.core.errors.NumbaPerformanceWarning


Writing pytest.ini


In [ ]:
%%writefile tests/conftest.py
"""The suite must run on a real CUDA GPU, never the CUDA simulator."""
import os
import numpy as np
import pytest
from numba import cuda


def pytest_sessionstart(session):
    if os.environ.get('NUMBA_ENABLE_CUDASIM') == '1':
        raise pytest.UsageError('Real GPU required: disable NUMBA_ENABLE_CUDASIM.')
    if not cuda.is_available():
        raise pytest.UsageError('CUDA unavailable. In Colab choose Runtime > Change runtime type > T4 GPU.')


@pytest.fixture
def rng():
    return np.random.default_rng(20261015)


@pytest.fixture(autouse=True)
def synchronized_test():
    yield
    cuda.synchronize()


Writing tests/conftest.py


In [ ]:
%%writefile tests/test_module1.py
"""Tasks 01–08: execute real CUDA kernels against independent NumPy results."""

import numpy as np
from numba import cuda
from lab_kernels import module1_execution_model as kernels


def test_task01_vector_add_1d(rng):
    for n in (1, 255, 256, 1024, 1_000_003):
        a = rng.normal(size=n).astype(np.float32)
        b = rng.normal(size=n).astype(np.float32)
        out = cuda.device_array(n, dtype=np.float32)
        kernels.vector_add_1d[(n + 255) // 256, 256](cuda.to_device(a), cuda.to_device(b), out)
        np.testing.assert_allclose(out.copy_to_host(), a + b, atol=1e-5)


def test_task02_vector_scale_grid_stride(rng):
    # A fixed grid has fewer threads than the longest input and must revisit
    # each lane's stride many times, including a partial final iteration.
    for n in (1, 1025, 10_000_003):
        x = rng.normal(size=n).astype(np.float32)
        alpha = np.float32(-2.25)
        out = cuda.device_array(n, dtype=np.float32)
        kernels.vector_scale_grid_stride[8, 128](cuda.to_device(x), alpha, out)
        np.testing.assert_allclose(out.copy_to_host(), x * alpha, atol=1e-5)


def test_task03_matrix_add_2d(rng):
    for shape in ((1, 1), (513, 1021)):
        a = rng.normal(size=shape).astype(np.float32)
        b = rng.normal(size=shape).astype(np.float32)
        out = cuda.device_array(shape, dtype=np.float32)
        blocks = ((shape[1] + 15) // 16, (shape[0] + 15) // 16)
        kernels.matrix_add_2d[blocks, (16, 16)](cuda.to_device(a), cuda.to_device(b), out)
        np.testing.assert_allclose(out.copy_to_host(), a + b, atol=1e-5)


def test_task04_matrix_transpose_naive(rng):
    for shape in ((1, 19), (17, 1), (513, 1021)):
        a = rng.normal(size=shape).astype(np.float32)
        out = cuda.device_array(shape[::-1], dtype=np.float32)
        blocks = ((shape[1] + 15) // 16, (shape[0] + 15) // 16)
        kernels.matrix_transpose_naive[blocks, (16, 16)](cuda.to_device(a), out)
        np.testing.assert_array_equal(out.copy_to_host(), a.T)


def test_task05_volume_init_3d():
    d, h, w = 33, 47, 65
    out = cuda.device_array((d, h, w), dtype=np.int32)
    kernels.volume_init_3d[((w + 7) // 8, (h + 7) // 8, (d + 7) // 8), (8, 8, 8)](out, d, h, w)
    z, y, x = np.indices((d, h, w), dtype=np.int32)
    expected = z * 10000 + y * 100 + x
    actual = out.copy_to_host()
    np.testing.assert_array_equal(actual, expected)
    # Explicitly validate C-order index flattening, including the last voxel.
    flat = actual.ravel()
    for zz, yy, xx in ((0, 0, 0), (1, 2, 3), (d - 1, h - 1, w - 1)):
        assert flat[zz * h * w + yy * w + xx] == zz * 10000 + yy * 100 + xx


def test_task06_rgb_to_grayscale(rng):
    for dtype in (np.uint8, np.float32):
        rgb = rng.integers(0, 256, size=(37, 65, 3), dtype=np.uint8).astype(dtype)
        out = cuda.device_array(rgb.shape[:2], dtype=np.float32)
        kernels.rgb_to_grayscale[(5, 3), (16, 16)](cuda.to_device(rgb), out)
        reference = 0.299 * rgb[..., 0] + 0.587 * rgb[..., 1] + 0.114 * rgb[..., 2]
        np.testing.assert_allclose(out.copy_to_host(), reference, atol=1e-4)


def test_task07_vector_lerp(rng):
    n = 100_003
    a, b = rng.normal(size=(2, n)).astype(np.float32)
    t = rng.uniform(-1, 2, size=n).astype(np.float32)
    t[:3] = (0, 1, 0.5)
    out = cuda.device_array(n, dtype=np.float32)
    kernels.vector_lerp[(n + 255) // 256, 256](cuda.to_device(a), cuda.to_device(b), cuda.to_device(t), out)
    np.testing.assert_allclose(out.copy_to_host(), (1 - t) * a + t * b, atol=1e-5)


def test_task08_vector_threshold(rng):
    x = rng.normal(size=8197).astype(np.float32)
    threshold = np.float32(0.25)
    x[:5] = (threshold, np.nextafter(threshold, -np.inf, dtype=np.float32), np.nextafter(threshold, np.inf, dtype=np.float32), -np.inf, np.inf)
    for dtype in (np.uint8, np.bool_):
        out = cuda.device_array(x.size, dtype=dtype)
        kernels.vector_threshold[3, 128](cuda.to_device(x), threshold, out)
        np.testing.assert_array_equal(out.copy_to_host(), (x >= threshold).astype(dtype))


Writing tests/test_module1.py


In [ ]:
%%writefile tests/test_module2.py
"""Tasks 09–17: GPU shared-memory correctness and real event measurements."""

import numpy as np
from numba import cuda
from scipy.ndimage import uniform_filter1d
from lab_kernels import module2_memory_hierarchy as kernels


def test_task09_benchmark_strided_access():
    n = 8197
    for stride in (1, 2, 4, 8, 16, 32):
        source = np.arange(n * stride, dtype=np.float32)
        out = cuda.device_array(n, dtype=np.float32)
        kernels.benchmark_strided_access[(n + 255) // 256, 256](cuda.to_device(source), out, stride)
        np.testing.assert_array_equal(out.copy_to_host(), source[::stride])
    measured = kernels.profile_strided_access(num_elements=1 << 16, repeats=5)
    assert set(measured) == {1, 2, 4, 8, 16, 32}
    for result in measured.values():
        assert result['integrity']
        assert result['elapsed_ms'] > 0
        assert np.isfinite(result['effective_gb_s']) and result['effective_gb_s'] > 0


def test_task10_stencil_1d_shared(rng):
    for n, radius in ((1, 3), (257, 0), (1027, 1), (8197, 7), (8197, 32)):
        x = rng.normal(size=n).astype(np.float32)
        out = cuda.device_array_like(x)
        kernels.stencil_1d_shared[(n + 255) // 256, 256](cuda.to_device(x), out, radius)
        reference = uniform_filter1d(x, size=2 * radius + 1, mode='nearest')
        np.testing.assert_allclose(out.copy_to_host(), reference, atol=1e-4)


def test_task11_conv_1d_shared_tiled(rng):
    for taps in (1, 4, 9, 64, 65):
        x = rng.normal(size=2051).astype(np.float32)
        weights = rng.normal(size=taps).astype(np.float32)
        out = cuda.device_array_like(x)
        kernels.conv_1d_shared_tiled[(x.size + 255) // 256, 256](cuda.to_device(x), cuda.to_device(weights), out)
        np.testing.assert_allclose(out.copy_to_host(), np.convolve(x, weights, mode='same'), atol=1e-4)


def test_task12_matmul_tiled_shared(rng):
    for m, k, n in ((1, 1, 1), (31, 19, 37), (64, 64, 64), (500, 500, 500), (1024, 1024, 1024)):
        a = rng.normal(size=(m, k)).astype(np.float32)
        b = rng.normal(size=(k, n)).astype(np.float32)
        out = cuda.device_array((m, n), dtype=np.float32)
        kernels.matmul_tiled_shared[((n + 15) // 16, (m + 15) // 16), (16, 16)](cuda.to_device(a), cuda.to_device(b), out)
        np.testing.assert_allclose(out.copy_to_host(), a @ b, atol=1e-3, rtol=1e-4)


def test_task13_transpose_bank_conflict_free(rng):
    for m, n in ((1, 1), (1, 35), (35, 1), (67, 97), (513, 1021)):
        a = rng.normal(size=(m, n)).astype(np.float32)
        out = cuda.device_array((n, m), dtype=np.float32)
        kernels.transpose_bank_conflict_free[((n + 31) // 32, (m + 31) // 32), (32, 8)](cuda.to_device(a), out)
        np.testing.assert_array_equal(out.copy_to_host(), a.T)


def test_task14_reduction_sum_shared(rng):
    for n, threads in ((1, 32), (257, 256), (1_000_003, 256), (2051, 1024)):
        x = rng.uniform(-1, 2, size=n).astype(np.float32)
        blocks = (n + threads - 1) // threads
        sums = cuda.device_array(blocks, dtype=np.float32)
        kernels.reduction_sum_shared[blocks, threads](cuda.to_device(x), sums)
        actual = sums.copy_to_host()
        reference_blocks = np.asarray([x[start:start + threads].sum(dtype=np.float64) for start in range(0, n, threads)])
        np.testing.assert_allclose(actual, reference_blocks, atol=2e-5, rtol=1e-5)
        np.testing.assert_allclose(actual.sum(dtype=np.float64), x.sum(dtype=np.float64), rtol=1e-4)


def test_task15_reduction_argmax_shared(rng):
    for n, threads in ((1, 32), (1031, 256), (2051, 1024)):
        x = rng.normal(size=n).astype(np.float32)
        # Equal maxima in the first and last blocks must preserve the earliest.
        x[0] = 99
        x[-1] = 99
        blocks = (n + threads - 1) // threads
        values = cuda.device_array(blocks, dtype=np.float32)
        indices = cuda.device_array(blocks, dtype=np.int64)
        kernels.reduction_argmax_shared[blocks, threads](cuda.to_device(x), values, indices)
        actual_values, actual_indices = values.copy_to_host(), indices.copy_to_host()
        for block in range(blocks):
            start = block * threads
            local_index = int(np.argmax(x[start:start + threads]))
            assert actual_indices[block] == start + local_index
            assert actual_values[block] == x[start + local_index]
        winner = int(np.argmax(actual_values))
        assert actual_indices[winner] == np.argmax(x)
    x = np.full(259, -np.inf, dtype=np.float32)
    values = cuda.device_array(2, dtype=np.float32)
    indices = cuda.device_array(2, dtype=np.int64)
    kernels.reduction_argmax_shared[2, 256](cuda.to_device(x), values, indices)
    np.testing.assert_array_equal(values.copy_to_host(), [-np.inf, -np.inf])
    np.testing.assert_array_equal(indices.copy_to_host(), [0, 256])


def test_task16_constant_memory_filter(rng):
    assert kernels.FIR_WEIGHTS.shape == (64,)
    for n in (64, 257, 8197):
        x = rng.normal(size=n).astype(np.float32)
        out = cuda.device_array_like(x)
        kernels.constant_memory_filter[(n + 255) // 256, 256](cuda.to_device(x), out)
        reference = np.convolve(x, kernels.FIR_WEIGHTS, mode='same')
        np.testing.assert_allclose(out.copy_to_host(), reference, atol=1e-4)


def test_task17_benchmark_pinned_transfer():
    result = kernels.benchmark_pinned_transfer(8 * 1024 * 1024, repeats=5)
    assert result['size_bytes'] == 8 * 1024 * 1024
    for kind in ('pageable', 'pinned'):
        assert result[kind]['integrity']
        for direction in ('h2d', 'd2h'):
            assert result[kind][direction]['elapsed_ms'] > 0
            assert np.isfinite(result[kind][direction]['gb_s'])
            assert result[kind][direction]['gb_s'] > 0
    for ratio in result['speedup'].values():
        assert np.isfinite(ratio) and ratio > 0
    # Hardware-dependent bandwidth ratios are reported, never forged or used
    # as a correctness gate (Colab may share resources with other workloads).


Writing tests/test_module2.py


In [ ]:
%%writefile tests/test_module3.py
"""GPU tests for tasks 18–20; CPU references validate each public contract."""

import numpy as np
from numba import cuda

from lab_kernels.module3_warp_atomics import (
    scan_hillis_steele,
    warp_reduce_sum_shfl,
    warp_vote_analyzer,
)


def test_task18_warp_reduce_sum_shfl():
    rng = np.random.default_rng(18)
    for size, threads in [(1, 32), (31, 32), (257, 256), (4099, 128), (1025, 1024)]:
        values = rng.normal(size=size).astype(np.float32)
        blocks = (size + threads - 1) // threads
        output = cuda.device_array(blocks, dtype=np.float64)
        warp_reduce_sum_shfl[blocks, threads](cuda.to_device(values), output)
        actual = output.copy_to_host()
        expected = np.array([
            values[start:start + threads].sum(dtype=np.float64)
            for start in range(0, size, threads)
        ])
        np.testing.assert_allclose(actual, expected, rtol=1e-6, atol=1e-7)
        np.testing.assert_allclose(actual.sum(), values.sum(dtype=np.float64), atol=1e-7)


def test_task19_warp_vote_analyzer():
    values = np.concatenate([
        np.ones(32),
        -np.ones(32),
        np.tile([-1, 1], 16),
        np.array([1, 0, -1, 2, 3], dtype=np.float32),
    ]).astype(np.float32)
    expected = []
    for start in range(0, values.size, 32):
        predicates = values[start:start + 32] > 0
        ballot = sum(int(flag) << lane for lane, flag in enumerate(predicates))
        expected.append([ballot, int(predicates.all()), int(predicates.any())])
    expected = np.asarray(expected, dtype=np.uint32)
    device_values = cuda.to_device(values)
    output = cuda.device_array(expected.shape, dtype=np.uint32)
    warp_vote_analyzer[1, 128](device_values, output)
    np.testing.assert_array_equal(output.copy_to_host(), expected)

    # The compact signature keeps the original ballot-only interpretation.
    ballots = cuda.device_array(expected.shape[0], dtype=np.uint32)
    warp_vote_analyzer[2, 64](device_values, ballots)
    np.testing.assert_array_equal(ballots.copy_to_host(), expected[:, 0])


def test_task20_scan_hillis_steele():
    rng = np.random.default_rng(20)
    for size in [1, 2, 31, 33, 255, 513, 1024]:
        values = rng.uniform(-1, 1, size=size).astype(np.float32)
        output = cuda.device_array(size, dtype=np.float32)
        threads = 1 << (size - 1).bit_length()
        scan_hillis_steele[1, threads](cuda.to_device(values), output)
        np.testing.assert_allclose(
            output.copy_to_host(), np.cumsum(values, dtype=np.float64),
            rtol=1e-6, atol=1e-5,
        )


Writing tests/test_module3.py


In [ ]:
%%writefile autograder.py
"""Run the included GPU correctness suite and save verifiable execution evidence.

This is a student self-check harness, not the instructor's private autograder.
"""
import argparse
import hashlib
import importlib.metadata
import json
import os
from pathlib import Path
import platform
import re
import subprocess
import sys
from datetime import datetime, timezone

ROOT = Path(__file__).resolve().parent


def source_hashes():
    paths = sorted((ROOT / 'lab_kernels').glob('*.py')) + sorted((ROOT / 'tests').glob('*.py'))
    paths += [ROOT / 'autograder.py', ROOT / 'requirements.txt']
    return {str(p.relative_to(ROOT)): hashlib.sha256(p.read_bytes()).hexdigest() for p in paths}


def gpu_info():
    if os.environ.get('NUMBA_ENABLE_CUDASIM') == '1':
        raise RuntimeError('GPU simulator is not accepted as hardware execution.')
    from numba import cuda
    if not cuda.is_available():
        raise RuntimeError('No CUDA GPU. Select a T4 GPU runtime in Colab.')
    d = cuda.get_current_device()
    name = d.name.decode() if isinstance(d.name, bytes) else d.name
    return {'name': name, 'compute_capability': list(d.compute_capability),
            'warp_size': d.WARP_SIZE, 'max_threads_per_block': d.MAX_THREADS_PER_BLOCK,
            'shared_memory_per_block': d.MAX_SHARED_MEMORY_PER_BLOCK,
            'driver_version': subprocess.check_output(['nvidia-smi', '--query-gpu=driver_version', '--format=csv,noheader'], text=True).strip()}


class ExecutionReport:
    def __init__(self):
        self.collected = []
        self.results = []

    def pytest_collection_finish(self, session):
        self.collected = [item.nodeid for item in session.items]

    def pytest_runtest_logreport(self, report):
        if report.when == 'call' or (report.when in ('setup', 'teardown') and report.failed):
            self.results.append({'test': report.nodeid, 'phase': report.when,
                                 'outcome': report.outcome, 'seconds': report.duration,
                                 'diagnostic': str(report.longrepr) if report.failed else None,
                                 'stdout': report.capstdout})


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument('--module', choices=[str(i) for i in range(1, 4)])
    args = parser.parse_args()
    os.chdir(ROOT)
    result_dir = ROOT / 'results'
    result_dir.mkdir(exist_ok=True)
    info = gpu_info()
    print(json.dumps(info, indent=2), flush=True)
    import pytest
    plugin = ExecutionReport()
    target = f'tests/test_module{args.module}.py' if args.module else 'tests'
    code = int(pytest.main([target, '-v', '--tb=short', f'--junitxml={result_dir / "junit.xml"}'], plugins=[plugin]))
    tasks = sorted({int(m.group(1)) for node in plugin.collected
                    if (m := re.search(r'test_task_?(\d+)', node))})
    passed = sum(r['outcome'] == 'passed' for r in plugin.results)
    report = {'generated_at_utc': datetime.now(timezone.utc).isoformat(),
              'student_id': '230103188', 'runtime': 'Google Colab' if 'COLAB_RELEASE_TAG' in os.environ else platform.platform(),
              'python': sys.version, 'gpu': info,
              'packages': {p: importlib.metadata.version(p) for p in ('numpy', 'numba', 'numba-cuda', 'scipy', 'pytest')},
              'exit_code': code, 'collected_tests': len(plugin.collected),
              'passed_tests': passed, 'task_ids': tasks,
              'all_20_tasks_passed': code == 0 and tasks == list(range(1, 21)) and passed == len(plugin.collected),
              'source_sha256': source_hashes(), 'tests': plugin.results,
              'scope': 'Included GPU correctness tests. Performance targets require separate hardware profiling; not instructor certification.'}
    (result_dir / 'execution_report.json').write_text(json.dumps(report, indent=2))
    (result_dir / 'environment.txt').write_text(subprocess.check_output(['nvidia-smi'], text=True) + '\n' + sys.version + '\n' + subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))
    print(f'\nTask coverage: {len(tasks)}/20; passed cases: {passed}/{len(plugin.collected)}; exit code: {code}', flush=True)
    if not args.module and code == 0 and not report['all_20_tasks_passed']:
        raise SystemExit('Incomplete task coverage: expected tasks 1 through 20.')
    raise SystemExit(code)


if __name__ == '__main__':
    main()


Writing autograder.py


## Run all 20 tasks on the GPU
The run saves a JSON report with task coverage, each test result, GPU details, package versions, and hashes of the executed sources.
This is the included student correctness suite; the instructor's private tests and hardware-counter profiling are separate.


In [ ]:
import json, subprocess, sys
completed = subprocess.run([sys.executable, '-u', 'autograder.py'], text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(completed.stdout)
(WORK / 'results' / 'verification_output.txt').write_text(completed.stdout)
assert completed.returncode == 0, 'GPU test suite failed; inspect the output above.'
report = json.loads((WORK / 'results' / 'execution_report.json').read_text())
assert report['all_20_tasks_passed'], 'Expected passing task coverage 1–20.'
print(f"SUCCESS: {len(report['task_ids'])}/20 tasks; {report['passed_tests']}/{report['collected_tests']} test cases passed on {report['gpu']['name']}.")


{
  "name": "Tesla T4",
  "compute_capability": [
    7,
    5
  ],
  "warp_size": 32,
  "max_threads_per_block": 1024,
  "shared_memory_per_block": 49152,
  "driver_version": "580.82.07"
}
============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/cuda_tasks_01_20
configfile: pytest.ini
plugins: typeguard-4.6.0, langsmith-0.14.2, anyio-4.15.1, platformdirs-4.12.2
collecting ... collected 20 items

tests/test_module1.py::test_task01_vector_add_1d PASSED                  [  5%]
tests/test_module1.py::test_task02_vector_scale_grid_stride PASSED       [ 10%]
tests/test_module1.py::test_task03_matrix_add_2d PASSED                  [ 15%]
tests/test_module1.py::test_task04_matrix_transpose_naive PASSED         [ 20%]
tests/test_module1.py::test_task05_volume_init_3d PASSED                 [ 25%]
tests/test_module1.py::test_task06_rgb_to_grayscale P

## Inspect and download the execution evidence
The ZIP contains the exact kernel and test sources plus the generated reports. No instructor HMAC token is claimed because no signing key or private protocol was supplied.


In [ ]:
from IPython.display import display
import pandas as pd
summary = [{'test': t['test'].split('::')[-1], 'outcome': t['outcome'], 'seconds': round(t['seconds'], 4)} for t in report['tests']]
display(pd.DataFrame(summary))
print('GPU:', report['gpu'])
print('Full report:', WORK / 'results' / 'execution_report.json')
import shutil
archive = shutil.make_archive('/content/CUDA_Tasks_01_20_results', 'zip', WORK)
from google.colab import files
files.download(archive)


,test,outcome,seconds
0,test_task01_vector_add_1d,passed,2.9529
1,test_task02_vector_scale_grid_stride,passed,1.1021
2,test_task03_matrix_add_2d,passed,0.4464
3,test_task04_matrix_transpose_naive,passed,0.3012
4,test_task05_volume_init_3d,passed,0.2645
5,test_task06_rgb_to_grayscale,passed,0.6076
6,test_task07_vector_lerp,passed,0.2508
7,test_task08_vector_threshold,passed,0.4628
8,test_task09_benchmark_strided_access,passed,0.6550
9,test_task10_stencil_1d_shared,passed,0.6598


GPU: {'name': 'Tesla T4', 'compute_capability': [7, 5], 'warp_size': 32, 'max_threads_per_block': 1024, 'shared_memory_per_block': 49152, 'driver_version': '580.82.07'}
Full report: /content/cuda_tasks_01_20/results/execution_report.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>